In [1]:
from langgraph.graph import START, END, StateGraph
from langgraph.types import Send
from typing import TypedDict
import subprocess
from openai import OpenAI
import textwrap
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator

llm = init_chat_model("openai:gpt-5-nano")


class State(TypedDict):

    video_file: str
    audio_file: str
    transcription: str
    summaries: Annotated[list[str], operator.add]

In [2]:
def extract_audio(state: State):
    # ffmpeg
    output_file = state["video_file"].replace("mp4", "mp3")
    command = [
        "ffmpeg",
        "-i",
        state["video_file"],
        "-filter:a",
        "atempo=2.0",
        "-y",  # override
        output_file,
    ]
    subprocess.run(command)
    return {"audio_file": output_file}


def transcribe_audio(state: State):
    # use audio file
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_f:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_f,
            language="ko",
            # prompt="Maple, Maple Story, Nexon, Unreal",
            prompt="메이플스토리, 러셀, 언리얼엔진",
        )
        return {"transcription": transcription}


def dispatch_summarizers(state: State):
    transcription = state["transcription"]
    chunks = []
    for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
        chunks.append({"id": i + 1, "chunk": chunk})

    return [Send("summarize_chunk", chunk) for chunk in chunks]


def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Summarize the following text.

        Text: {chunk}
        """
    )
    summary = f"[Chunk: {chunk_id} {response.content}]"
    return {"summaries": [summary]}

In [3]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges(
    "transcribe_audio", dispatch_summarizers, ["summarize_chunk"]
)
graph_builder.add_edge("summarize_chunk", END)

graph = graph_builder.compile()

In [4]:
graph.invoke({"video_file": "temp.mp4"})

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex

{'video_file': 'temp.mp4',
 'audio_file': 'temp.mp3',
 'transcription': '안녕하세요 여러분, 러셀입니다. 여러분들은 모두 게임에 대한 추억이 있으신가요? 저는 어렸을 때 게임 메이플스토리에서 넓은 맵을 돌아다니고 몬스터를 피해 헤맸던 것이 마치 제가 직접 모험하는 느낌이 나서 게임을 즐기는 내내 설레었던 추억이 있습니다. 하지만 시간이 지나면서 게임에 여러 변화가 있었는데 그 사이에서 모험하는 느낌과 설레임은 어느샌가 사라지고 없어져 많이 아쉬웠습니다. 그래서 익숙하지만 이제 많이 잊혀진 추억의 메이플스토리 맵을 언리얼엔진으로 재보장하여 여러분들과 같이 직접 체험하면서 추억을 회상하고자 이 프로젝트를 시작했습니다. 그 첫번째는 바로 시간의 신전입니다. 게임을 시작하면 신전에 들어가기 전 입구 맵부터 시작합니다. 리플레이에서 날아온 후 처음 마주하는 곳이죠. 조작키는 메이플스토리와 거의 같습니다. 방향키로 움직이고 알트키로 점프할 수 있습니다. 알트를 두번 누르면 더블점프도 할 수 있어요. 옛날 메이플처럼 넣지 않을까 하다가 너무 답답할 것 같아 넣었습니다. 그래도 천천히 걸어다니시면서 감상하시는 걸 추천해드려요. 오른쪽으로 가서 계단을 통해 걸어 올라가면 시간의 신전이 나옵니다. 저는 이 신전을 만들 때 신전에 고요하고 고급스러운 분위기에 초점을 두고 제작했습니다. 여신상과 수호대장 모델링도 최대한 잘 어울리게 모델링을 제작했죠. 조금 더 옆으로 가보면 신전의 가장 핵심적인 세계의 문이 나옵니다. 원래는 미래의 문, 현재의 문도 모두 열려있지만 지금은 과거의 문으로만 이동이 가능합니다. 이 문들도 언젠간 열리겠죠. 과거의 문으로 들어가면 추억의 길로 이동할 수 있습니다. 하지만 이 작품에서는 여러분들이 편하게 둘러보실 수 있도록 추억의 길, 후의 길, 그리고 망각의 길으로 이동할 수 있는 포탈을 한 맵에 제작했습니다. 절대 하나하나 만들기 귀찮습니다. 이번에는 신전의 세계의 문을 만들겠습니다. 신전의 세